# 07 — Depth Analysis

This notebook digs deeper into the results already produced by `05_model_development.ipynb` and `06_leakage_experiment.ipynb`, using **only the saved validation predictions** — no model is retrained, and the test set is never read.

Three analyses:

1. **Paired bootstrap** (1,000 resamples) for the Average Precision difference caused by adding `duration`, with 95% confidence intervals, per model and split.
2. **Threshold sensitivity** for the realistic (no-`duration`) Logistic Regression and Random Forest models — precision/recall/F1 across thresholds 0.05–0.95, compared to the default 0.5.
3. **Failure-mode audit** of the realistic models — false positive/false negative rates overall and by `job`/`poutcome` subgroup (subgroups with fewer than 100 validation rows are excluded as too small to read reliably).

All three use **validation data only**, and every comparison explicitly verifies that predictions are aligned (same `row_index`, same `y_true`) before being compared.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

%matplotlib inline
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42
RESULTS_DIR = "../results"
DEPTH_DIR = f"{RESULTS_DIR}/depth"
os.makedirs(DEPTH_DIR, exist_ok=True)

## Load saved validation predictions (read-only)

- `results/validation_predictions.csv` — Phase 5, realistic (no-`duration`) models: `majority_baseline`, `logistic_regression`, `random_forest`.
- `results/leakage_predictions.csv` — Phase 6, with-`duration` models: `logistic_regression`, `random_forest`.
- The raw dataset is loaded **only** to attach `job`/`poutcome` to validation rows for the Part 3 subgroup audit — no feature is recomputed and no model touches it.

None of these files are modified in this notebook.

In [ ]:
no_duration_path = f"{RESULTS_DIR}/validation_predictions.csv"
with_duration_path = f"{RESULTS_DIR}/leakage_predictions.csv"
for path in [no_duration_path, with_duration_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found — run 05_model_development.ipynb and 06_leakage_experiment.ipynb first.")

no_duration_preds = pd.read_csv(no_duration_path)
with_duration_preds = pd.read_csv(with_duration_path)

REALISTIC_MODELS = ["logistic_regression", "random_forest"]
SPLITS = ["stratified", "chronological"]

# Raw dataset, read-only, only to attach job/poutcome by row position for Part 3.
CSV_NAME = "bank-additional-full.csv"
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
except ImportError:
    pass
_candidates = [
    f"../data/{CSV_NAME}", f"data/{CSV_NAME}",
    f"/content/drive/MyDrive/bank-marketing-leakage-audit/data/{CSV_NAME}",
    f"/content/drive/MyDrive/{CSV_NAME}",
]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), None)
if DATA_PATH is None:
    try:
        from google.colab import files  # type: ignore
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Could not find '{CSV_NAME}' in any of: {_candidates}")

df_raw = pd.read_csv(DATA_PATH, sep=";")

print("no_duration_preds:", no_duration_preds.shape, no_duration_preds["model"].unique())
print("with_duration_preds:", with_duration_preds.shape, with_duration_preds["model"].unique())
print("df_raw:", df_raw.shape)

## Part 1: Paired bootstrap for the AP difference (with vs. without `duration`)

For each (split, model), we take the **same validation rows** scored by both the no-`duration` and with-`duration` model, verify they're perfectly aligned (same `row_index` set, same `y_true` per row), then run a **paired** bootstrap: each of the 1,000 resamples draws row indices with replacement *once* and applies that same resample to both score vectors, so the pairing (and therefore the variance reduction from pairing) is preserved. The 95% CI is the 2.5th/97.5th percentile of the 1,000 resampled ΔAP values.

In [ ]:
def get_aligned_pair(split_name, model_name):
    """Return (row_index, y_true_bin, proba_no_duration, proba_with_duration), verified aligned."""
    no_dur = (no_duration_preds[(no_duration_preds["split_strategy"] == split_name) &
                                 (no_duration_preds["model"] == model_name)]
              .set_index("row_index").sort_index())
    with_dur = (with_duration_preds[(with_duration_preds["split_strategy"] == split_name) &
                                     (with_duration_preds["model"] == model_name)]
                .set_index("row_index").sort_index())

    assert len(no_dur) > 0 and len(with_dur) > 0, f"Missing predictions for {split_name}/{model_name}"
    assert list(no_dur.index) == list(with_dur.index), \
        f"row_index sets differ between no-duration and with-duration predictions for {split_name}/{model_name}"
    assert (no_dur["y_true"].values == with_dur["y_true"].values).all(), \
        f"y_true mismatch between no-duration and with-duration predictions for {split_name}/{model_name}"

    row_index = no_dur.index.values
    y_true_bin = (no_dur["y_true"].values == "yes").astype(int)
    proba_no = no_dur["y_proba_yes"].values
    proba_with = with_dur["y_proba_yes"].values
    return row_index, y_true_bin, proba_no, proba_with


def paired_bootstrap_ap_diff(y_true_bin, proba_no, proba_with, n_boot=1000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n = len(y_true_bin)
    deltas = np.empty(n_boot)
    for b in range(n_boot):
        sample_idx = rng.integers(0, n, size=n)  # same resample applied to both arms -> paired
        yb = y_true_bin[sample_idx]
        ap_no = average_precision_score(yb, proba_no[sample_idx])
        ap_with = average_precision_score(yb, proba_with[sample_idx])
        deltas[b] = ap_with - ap_no
    return deltas


bootstrap_rows = []
for split_name in SPLITS:
    for model_name in REALISTIC_MODELS:
        row_index, y_true_bin, proba_no, proba_with = get_aligned_pair(split_name, model_name)

        observed_ap_no = average_precision_score(y_true_bin, proba_no)
        observed_ap_with = average_precision_score(y_true_bin, proba_with)
        observed_delta = observed_ap_with - observed_ap_no

        deltas = paired_bootstrap_ap_diff(y_true_bin, proba_no, proba_with, n_boot=1000, seed=RANDOM_STATE)
        ci_lower, ci_upper = np.percentile(deltas, [2.5, 97.5])

        bootstrap_rows.append({
            "split_strategy": split_name,
            "model": model_name,
            "n_validation": len(row_index),
            "ap_no_duration": observed_ap_no,
            "ap_with_duration": observed_ap_with,
            "observed_delta_ap": observed_delta,
            "ci_lower_2.5": ci_lower,
            "ci_upper_97.5": ci_upper,
            "n_resamples": 1000,
        })
        print(f"[{split_name:13s} | {model_name:20s}] "
              f"ΔAP = {observed_delta:+.4f}  95% CI [{ci_lower:+.4f}, {ci_upper:+.4f}]")

bootstrap_df = pd.DataFrame(bootstrap_rows)
bootstrap_df

In [ ]:
bootstrap_path = f"{DEPTH_DIR}/bootstrap_ap_difference.csv"
bootstrap_df.to_csv(bootstrap_path, index=False)
print("Saved:", bootstrap_path)

## Part 2: Threshold sensitivity (realistic Logistic Regression / Random Forest)

Using the saved predicted probabilities for the **no-`duration`** (realistic) models, we recompute precision/recall/F1 at thresholds from 0.05 to 0.95 (step 0.05, which includes the default 0.5), for each split. This shows how much each model's precision/recall trade-off can be shifted by threshold choice alone, without retraining anything.

In [ ]:
THRESHOLDS = np.round(np.arange(0.05, 1.00, 0.05), 2)
assert 0.5 in THRESHOLDS

threshold_rows = []
for split_name in SPLITS:
    for model_name in REALISTIC_MODELS:
        subset = no_duration_preds[(no_duration_preds["split_strategy"] == split_name) &
                                    (no_duration_preds["model"] == model_name)]
        y_true_bin = (subset["y_true"].values == "yes").astype(int)
        proba = subset["y_proba_yes"].values

        for t in THRESHOLDS:
            y_pred_t = (proba >= t).astype(int)
            threshold_rows.append({
                "split_strategy": split_name,
                "model": model_name,
                "threshold": t,
                "precision": precision_score(y_true_bin, y_pred_t, zero_division=0),
                "recall": recall_score(y_true_bin, y_pred_t, zero_division=0),
                "f1": f1_score(y_true_bin, y_pred_t, zero_division=0),
                "is_default_threshold": bool(np.isclose(t, 0.5)),
            })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df[threshold_df["is_default_threshold"]]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True)

for row, split_name in enumerate(SPLITS):
    for col, model_name in enumerate(REALISTIC_MODELS):
        ax = axes[row, col]
        sub = threshold_df[(threshold_df["split_strategy"] == split_name) &
                            (threshold_df["model"] == model_name)].sort_values("threshold")
        ax.plot(sub["threshold"], sub["precision"], marker="o", label="Precision", color="#4C72B0")
        ax.plot(sub["threshold"], sub["recall"], marker="o", label="Recall", color="#DD8452")
        ax.plot(sub["threshold"], sub["f1"], marker="o", label="F1", color="#55A868")
        ax.axvline(0.5, color="gray", linestyle="--", alpha=0.6, label="default (0.5)")
        ax.set_title(f"{split_name} — {model_name}")
        ax.set_xlabel("Threshold")
        ax.set_ylabel("Score")
        ax.grid(alpha=0.3)

axes[0, 0].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig(f"{DEPTH_DIR}/threshold_sensitivity.png", dpi=150)
plt.show()

In [ ]:
threshold_path = f"{DEPTH_DIR}/threshold_sensitivity.csv"
threshold_df.to_csv(threshold_path, index=False)
print("Saved:", threshold_path)
print("Saved:", f"{DEPTH_DIR}/threshold_sensitivity.png")

## Part 3: Failure-mode audit (realistic models)

For each realistic model and split, we classify every validation row as TP/FP/FN/TN at the default 0.5 threshold (using the saved `y_pred`, which was produced at that same threshold), then:

1. Report overall FP/FN counts and rates (false positive rate = FP / actual negatives; false negative rate = FN / actual positives).
2. Break those down by `job` and by `poutcome`, joined back from the raw dataset by `row_index` — **only subgroups with at least 100 validation rows** are reported, since smaller subgroups give unreliable rate estimates.

This is descriptive only — it characterizes *where* each model tends to err, not a fairness/bias determination.

In [ ]:
MIN_SUBGROUP_SIZE = 100


def classify_outcome(y_true, y_pred):
    outcome = np.full(len(y_true), "TN", dtype=object)
    outcome[(y_true == "yes") & (y_pred == "yes")] = "TP"
    outcome[(y_true == "no") & (y_pred == "yes")] = "FP"
    outcome[(y_true == "yes") & (y_pred == "no")] = "FN"
    return outcome


def subgroup_error_table(merged, group_col, min_size=MIN_SUBGROUP_SIZE):
    rows = []
    for group_value, g in merged.groupby(group_col):
        n = len(g)
        if n < min_size:
            continue
        n_pos = (g["y_true"] == "yes").sum()
        n_neg = (g["y_true"] == "no").sum()
        fp = (g["outcome"] == "FP").sum()
        fn = (g["outcome"] == "FN").sum()
        rows.append({
            group_col: group_value,
            "n": n,
            "n_actual_positive": int(n_pos),
            "n_actual_negative": int(n_neg),
            "fp_count": int(fp),
            "fn_count": int(fn),
            "fp_rate": fp / n_neg if n_neg > 0 else np.nan,
            "fn_rate": fn / n_pos if n_pos > 0 else np.nan,
        })
    return pd.DataFrame(rows).sort_values("n", ascending=False).reset_index(drop=True)


overall_rows = []
job_subgroup_rows = []
poutcome_subgroup_rows = []

for split_name in SPLITS:
    for model_name in REALISTIC_MODELS:
        subset = no_duration_preds[(no_duration_preds["split_strategy"] == split_name) &
                                    (no_duration_preds["model"] == model_name)].copy()
        subset = subset.merge(
            df_raw[["job", "poutcome"]], left_on="row_index", right_index=True, how="left", validate="one_to_one"
        )
        subset["outcome"] = classify_outcome(subset["y_true"].values, subset["y_pred"].values)

        n = len(subset)
        n_pos = (subset["y_true"] == "yes").sum()
        n_neg = (subset["y_true"] == "no").sum()
        tp = (subset["outcome"] == "TP").sum()
        fp = (subset["outcome"] == "FP").sum()
        fn = (subset["outcome"] == "FN").sum()
        tn = (subset["outcome"] == "TN").sum()
        overall_rows.append({
            "split_strategy": split_name, "model": model_name,
            "n": n, "n_actual_positive": int(n_pos), "n_actual_negative": int(n_neg),
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
            "fp_rate": fp / n_neg if n_neg > 0 else np.nan,
            "fn_rate": fn / n_pos if n_pos > 0 else np.nan,
        })

        job_table = subgroup_error_table(subset, "job")
        job_table.insert(0, "model", model_name)
        job_table.insert(0, "split_strategy", split_name)
        job_subgroup_rows.append(job_table)

        poutcome_table = subgroup_error_table(subset, "poutcome")
        poutcome_table.insert(0, "model", model_name)
        poutcome_table.insert(0, "split_strategy", split_name)
        poutcome_subgroup_rows.append(poutcome_table)

failure_overall_df = pd.DataFrame(overall_rows)
failure_by_job_df = pd.concat(job_subgroup_rows, ignore_index=True)
failure_by_poutcome_df = pd.concat(poutcome_subgroup_rows, ignore_index=True)

failure_overall_df

In [ ]:
failure_by_job_df

In [ ]:
failure_by_poutcome_df

In [ ]:
overall_path = f"{DEPTH_DIR}/failure_mode_overall.csv"
by_job_path = f"{DEPTH_DIR}/failure_mode_by_job.csv"
by_poutcome_path = f"{DEPTH_DIR}/failure_mode_by_poutcome.csv"

failure_overall_df.to_csv(overall_path, index=False)
failure_by_job_df.to_csv(by_job_path, index=False)
failure_by_poutcome_df.to_csv(by_poutcome_path, index=False)

print("Saved:", overall_path)
print("Saved:", by_job_path, "-", failure_by_job_df.shape)
print("Saved:", by_poutcome_path, "-", failure_by_poutcome_df.shape)

## Summary

- **Paired bootstrap**: for every (split, model), alignment (same `row_index` set, same `y_true`) was verified before pairing; 1,000-resample 95% CIs for ΔAP are reported in `results/depth/bootstrap_ap_difference.csv` — see whether each interval excludes zero, which is the formal confirmation that the `duration`-driven AP jump from `06_leakage_experiment.ipynb` is not bootstrap noise.
- **Threshold sensitivity**: precision/recall/F1 across thresholds 0.05–0.95 for both realistic models and both splits, saved as a table and a 2×2 plot — useful for picking an operating point later, without retraining.
- **Failure-mode audit**: overall and `job`/`poutcome`-level FP/FN rates for both realistic models and both splits, restricted to subgroups with ≥100 validation rows.

All analysis used only the already-saved validation predictions; no model was retrained, no existing results file was modified, and the test set was never read.